In [23]:
import pandas as pd
import json

file_path = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_metrics.json'
with open(file_path, 'r') as f:
    evaluation_results = json.load(f)

df = pd.DataFrame(evaluation_results)
df

,scenario_id,brand_name,scores,conversation,timestamp,status,errors
0,full_conversation,Contoso Restaurant,"{'BrandVoiceMetric': 0.9, 'RelevanceMetric': 0...","[{'role': 'user', 'content': 'Hello'}, {'role'...",2025-06-30T10:43:28.871519,success,[]
1,full_conversation,Domino's Pizza,"{'BrandVoiceMetric': 0.3, 'RelevanceMetric': 0...","[{'role': 'user', 'content': 'Hello'}, {'role'...",2025-06-30T10:44:35.043610,success,[]
2,full_conversation,P.F. Chang’s,"{'BrandVoiceMetric': 0.2, 'RelevanceMetric': 0...","[{'role': 'user', 'content': 'Hello'}, {'role'...",2025-06-30T10:45:30.608298,success,[]
3,full_conversation,Chipotle,"{'BrandVoiceMetric': 0.3, 'RelevanceMetric': 0...","[{'role': 'user', 'content': 'Hello'}, {'role'...",2025-06-30T10:46:35.970090,success,[]


In [24]:
conversations =[]
for brand_data in evaluation_results:
    brand_name = brand_data['brand_name']
    # Extract conversation pairs
    for i in range(0, len(brand_data['conversation']), 2):
        if i + 1 < len(brand_data['conversation']):  
            user_msg = brand_data['conversation'][i]
            assistant_msg = brand_data['conversation'][i + 1]
            
            conversations.append({
                'brand_name': brand_name,
                'user_prompt': user_msg['content'],
                'assistant_response': assistant_msg['content']
            })

In [25]:
conversations_df = pd.DataFrame(conversations)
conversations_df.to_csv('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/conversations.csv', index=False)
conversations_df

,brand_name,user_prompt,assistant_response
0,Contoso Restaurant,Hello,Hello there!Welcome to Contoso Restaurant!We'r...
1,Contoso Restaurant,"Hello, I want to order dinner for my family","That sounds wonderful!At Contoso Restaurant,we..."
2,Contoso Restaurant,I want hamburger and cheeseburger with large f...,Great choice!Our hamburgers and cheeseburgers ...
3,Contoso Restaurant,Can you put together a meal for 4 people?,"Absolutely!Let’s create a delicious,home-style..."
4,Contoso Restaurant,Include medium size drinks for everyone,Got it!Here’s the updated meal for your family...
5,Contoso Restaurant,Increase the order by 2 people and give me a s...,Absolutely!Let’s adjust the order to serve six...
6,Contoso Restaurant,Can you summarize my order?,"Of course!Here's a summary of your delicious,h..."
7,Domino's Pizza,Hello,Hi there!😊 How can I assist you today?
8,Domino's Pizza,"Hello, I want to order dinner for my family",Great!What would you like to order?We’ve got b...
9,Domino's Pizza,I want hamburger and cheeseburger with large f...,Got it!Let me confirm your order:\n\n- 1 Hambu...


In [26]:
evaluations = []

for brand_data in evaluation_results: 
    eval_data = { 'brand_name': brand_data['brand_name'], 
                 'BrandVoiceMetric': brand_data['scores']['BrandVoiceMetric'], 
                 'RelevanceMetric': brand_data['scores']['RelevanceMetric'], 
                 'TaskCompletionMetric': brand_data['scores']['TaskCompletionMetric'] } 
    evaluations.append(eval_data)

evaluation_df = pd.DataFrame(evaluations) 
evaluation_df.to_csv('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_scores.csv', index=False)

#Evaluation

In [1]:
import os
from azure.ai.evaluation import RelevanceEvaluator, CoherenceEvaluator, FluencyEvaluator
from dotenv import load_dotenv
from azure.ai.evaluation._model_configurations import AzureOpenAIModelConfiguration


load_dotenv()

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

model_config = {
    "azure_endpoint": ENDPOINT,
    "azure_deployment": DEPLOYMENT_NAME,
    "api_key": API_KEY,
}

if not model_config.get("azure_endpoint"):
	raise ValueError("Missing required Azure endpoint in model_config.")
if not model_config.get("azure_deployment"):
	raise ValueError("Missing required Azure deployment in model_config.")
if not model_config.get("api_key"):
	raise ValueError("Missing required API key in model_config.")

azure_model_config = AzureOpenAIModelConfiguration(
	azure_endpoint=str(model_config["azure_endpoint"]),
	azure_deployment=str(model_config["azure_deployment"]),
	api_key=str(model_config["api_key"]),
	api_version=str(model_config.get("api_version", "2024-12-01-preview"))
)
relevance_eval = RelevanceEvaluator(model_config=azure_model_config, threshold=3)
coherence_eval = CoherenceEvaluator(model_config=azure_model_config, threshold=3) 
fluency_eval = FluencyEvaluator(model_config=azure_model_config, threshold=3)

test_query = "What is the capital of Japan?" 
test_response = "The capital of Japan is Tokyo."

print("Testing Relevance Evaluator:") 
relevance_result = relevance_eval(query=test_query, response=test_response) 
print(f"Relevance result: {relevance_result}")

print("\nTesting Coherence Evaluator:") 
coherence_result = coherence_eval(query=test_query, response=test_response) 
print(f"Coherence result: {coherence_result}")

print("\nTesting Fluency Evaluator:") 
fluency_result = fluency_eval(query=test_query, response=test_response) 
print(f"Fluency result: {fluency_result}")
   

Testing Relevance Evaluator:
Relevance result: {'relevance': 4.0, 'gpt_relevance': 4.0, 'relevance_reason': 'The RESPONSE fully and accurately answers the QUERY, making it a complete response.', 'relevance_result': 'pass', 'relevance_threshold': 3}

Testing Coherence Evaluator:
Coherence result: {'coherence': 4.0, 'gpt_coherence': 4.0, 'coherence_reason': 'The RESPONSE is clear, direct, and logically organized, making it a coherent answer to the QUERY.', 'coherence_result': 'pass', 'coherence_threshold': 3}

Testing Fluency Evaluator:
Fluency result: {'fluency': 3.0, 'gpt_fluency': 3.0, 'fluency_reason': 'The response is clear, grammatically correct, and easy to understand, but it lacks complexity and variety in sentence structure and vocabulary.', 'fluency_result': 'pass', 'fluency_threshold': 3}


In [12]:
import pandas as pd
file_path = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_4'
conversations_df = pd.read_csv(file_path + '/conversations.csv')


In [ ]:
import asyncio
import time

async def evaluate_conversations_multi_metrics(conversations_df, batch_size=2, delay_between_batches=60): 
    evaluation_results = [] # Add delay between API calls to respect rate limits 
    total_batches = len(conversations_df) // batch_size + (1 if len(conversations_df) % batch_size else 0)
    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(conversations_df))
        batch_df = conversations_df.iloc[start_idx:end_idx]
        
        print(f"Processing batch {batch_num + 1}/{total_batches} (rows {start_idx}-{end_idx-1})")
        
        for idx, row in batch_df.iterrows():
            try:
                # Add delay between each evaluation
                await asyncio.sleep(5)  # Increased delay for multiple metrics
                
                # Evaluate with Relevance
                relevance_result = relevance_eval(
                    query=row['user_prompt'],
                    response=row['assistant_response']
                )
                
                # Small delay between metrics
                await asyncio.sleep(2)
                
                # Evaluate with Coherence
                coherence_result = coherence_eval(
                    query=row['user_prompt'],
                    response=row['assistant_response']
                )
                
                # Small delay between metrics
                await asyncio.sleep(2)
                
                # Evaluate with Fluency
                fluency_result = fluency_eval(
                    query=row['user_prompt'],
                    response=row['assistant_response']
                )
                
                evaluation_results.append({
                    'brand_name': row['brand_name'],
                    'user_prompt': row['user_prompt'],
                    'assistant_response': row['assistant_response'],
                    'relevance_score': relevance_result['relevance'],
                    'coherence_score': coherence_result['coherence'],
                    'fluency_score': fluency_result['fluency'],
                    'turn_number': idx // 4 + 1
                })
                
                print(f"  Evaluated row {idx}: relevance={relevance_result['relevance']}, coherence={coherence_result['coherence']}, fluency={fluency_result['fluency']}")
                
            except Exception as e:
                print(f"Error evaluating conversation {idx}: {str(e)}")
                continue
        
        # Wait between batches (except for the last batch)
        if batch_num < total_batches - 1:
            print(f"Waiting {delay_between_batches} seconds before next batch...")
            await asyncio.sleep(delay_between_batches)
            
    return pd.DataFrame(evaluation_results)

In [ ]:
evaluation_results_df = await evaluate_conversations_multi_metrics(conversations_df, batch_size=2,
    delay_between_batches=60)

if not evaluation_results_df.empty:
    print("\nEvaluation Results Summary:")
    print("Average Scores by Brand:")
    summary_stats = evaluation_results_df.groupby('brand_name')[['relevance_score', 'coherence_score', 'fluency_score']].mean()
    print(summary_stats.round(2))
    
    
    evaluation_results_df.to_csv(file_path+'/conversation_metric_scores.csv', index=False)
    summary_stats.to_csv(file_path+'/average_scores_by_brand.csv', index=True)
    print("\nResults saved to: evaluation_results/conversation_multi_metric_scores.csv")
else:
    print("No results generated")

Processing batch 1/14 (rows 0-1)
  Evaluated row 0: relevance=3.0, coherence=3.0, fluency=4.0
  Evaluated row 1: relevance=4.0, coherence=4.0, fluency=4.0
Waiting 60 seconds before next batch...
Processing batch 2/14 (rows 2-3)
  Evaluated row 2: relevance=5.0, coherence=4.0, fluency=4.0
  Evaluated row 3: relevance=5.0, coherence=4.0, fluency=3.0
Waiting 60 seconds before next batch...
Processing batch 3/14 (rows 4-5)
  Evaluated row 4: relevance=5.0, coherence=4.0, fluency=4.0
  Evaluated row 5: relevance=5.0, coherence=4.0, fluency=4.0
Waiting 60 seconds before next batch...
Processing batch 4/14 (rows 6-7)
  Evaluated row 6: relevance=5.0, coherence=4.0, fluency=4.0
  Evaluated row 7: relevance=4.0, coherence=4.0, fluency=3.0
Waiting 60 seconds before next batch...
Processing batch 5/14 (rows 8-9)
  Evaluated row 8: relevance=4.0, coherence=4.0, fluency=3.0
  Evaluated row 9: relevance=5.0, coherence=4.0, fluency=4.0
Waiting 60 seconds before next batch...
Processing batch 6/14 (ro

In [26]:
import pandas as pd
file_path = 'C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_1/evaluation_scores.csv'
evaluation_results_df = pd.read_csv(file_path)

In [27]:
import plotly.express as px
import matplotlib.pyplot as plt

# Create summary by brand
brand_summary = evaluation_results_df.groupby('brand_name')[['BrandVoiceMetric', 'RelevanceMetric', 'TaskCompletionMetric']].mean().reset_index()

# Multi-metric comparison by brand
fig1 = px.bar(
    brand_summary.melt(id_vars='brand_name', var_name='metric', value_name='score'),
    x='brand_name',
    y='score',
    color='metric',
    barmode='group',
    title='Custom Evaluation Scores by Brand and Metric (temp:0.7, top_p:0.95)',
    width=800,  
    height=500  
)
fig1.update_layout(legend_title_text='custom_metric')  
# Save as HTML (this always works)
fig1.write_html('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_1/custom_brand_metric_comparison.html')
#fig1.write_image('C:/Users/t-toluale/.vscode/Ordering_ChatBot/evaluation_results/evaluation_1/brand_metric_comparison.png')
fig1.show()

In [ ]:
fig1 = px.bar( evaluation_results_df.groupby('brand_name')['relevance_score'].mean().reset_index(), 
              x='brand_name', y='relevance_score', 
              title='Average Relevance Score by Brand' ) 
fig1.show()

In [ ]:
fig3 = px.line( evaluation_results_df.groupby(['brand_name', 'turn_number'])['relevance_score'].mean().reset_index(), 
               x='turn_number', y='relevance_score', color='brand_name', 
               title='Relevance Scores Over Conversation Progress' ) 
fig3.show()